In [1]:
!pip install torch transformers

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [2]:
import json
import os
import torch
CLASSES = {
    "Adware": 0,
    "Backdoor": 1,
    "Botnet": 2,
    "CGI": 3,
    "Code-execution": 4,
    "DDos": 5,
    "Dir-Traversal": 6,
    "Dos": 7,
    "Info-Disclosure": 8,
    "Injection": 9,
    "Other": 10,
    "Overflow": 11,
    "Ransomware": 12,
    "Remote-file-Inclusion": 13,
    "Scanner": 14,
    "Spyware": 15,
    "Trojan": 16,
    "Virus": 17,
    "Webshell": 18,
    "Worm": 19,
    "XSS": 20
}
INV_CLASSES = {v: k for k, v in CLASSES.items()}
CONCEPTS= ["ip", "injection"]
CLASSES_TO_EXAMINE = ["Adware", "Scanner", "Spyware", "Trojan", "XSS", "Remote-file-Inclusion", "Overflow", "Injection", "Info-Disclosure", "Dir-Traversal", "Code-execution", "CGI", "Ransomware", "Botnet", "Backdoor"]
MODEL_NAME = "./models/codebert-base-mlm"


from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, output_hidden_states=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

d:\fabio\tesi\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
print(model)

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data as data
import numpy as np

In [ ]:
# Estrazione delle feature
f = open("./data/packet_inspection/packets_dataset.jsonl", "r")
dataset = [json.loads(line) for line in f.readlines()]

tokenized_inputs = tokenizer([s["text"] for s in dataset], padding=True, truncation=True, return_tensors="pt")
with torch.no_grad():
    outputs = model(**tokenized_inputs)
    hidden_states = outputs.hidden_states
    # Prendi l'ultimo layer nascosto
    activations = [h.numpy() for h in hidden_states]  # Converti in numpy array

Using CPU


In [6]:

class SAE(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(SAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim, input_dim),
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded, encoded
    

In [7]:
def calculate_sae_loss(x, W_enc, b_enc, W_dec, b_dec, lambd):
    """
    Calcola la loss function per uno Sparse AutoEncoder come specificato nell'immagine.

    Args:
        x (torch.Tensor): Tensore dei dati di input, di forma (batch_size, D).
        W_enc (torch.Tensor): Pesi dell'encoder, di forma (F, D).
        b_enc (torch.Tensor): Bias dell'encoder, di forma (F,).
        W_dec (torch.Tensor): Pesi del decoder, di forma (D, F).
        b_dec (torch.Tensor): Bias del decoder, di forma (D,).
        lambd (float): Parametro di regolarizzazione.

    Returns:
        torch.Tensor: Il valore scalare della loss.
    """
    # 1. Calcolo delle attivazioni dei feature (f_i(x) = ReLU(W_enc * x + b_enc))
    # Il prodotto matriciale (matmul) deve tenere conto delle dimensioni
    # Il documento indica W_enc come (F, D), quindi lo usiamo direttamente
    # per matmul(W_enc, x.T) o usiamo x con matmul(x, W_enc.T)
    feature_activations = torch.relu(torch.matmul(x, W_enc.T) + b_enc)

    # 2. Ricostruzione dell'input (x_hat = W_dec * f(x) + b_dec)
    # W_dec è di forma (D, F) e feature_activations di forma (batch_size, F)
    x_hat = torch.matmul(feature_activations, W_dec.T) + b_dec
    
    # 3. Calcolo del termine di ricostruzione (L2 norm)
    # ||x - x_hat||_2^2
    reconstruction_loss = torch.mean(torch.sum((x - x_hat)**2, dim=1))
    
    # 4. Calcolo del termine di penalizzazione (L1 norm)
    # lambda * sum(f_i(x) * ||W_dec_i||_2)
    # La tua immagine mostra un L2 norm sui pesi del decoder,
    # ma una regolarizzazione L1 sulle attivazioni.
    # Spieghiamola così: la somma dei valori assoluti delle attivazioni moltiplicata per il peso
    # e una lambda.
    # W_dec è (D,F), quindi W_dec_i (cioè W_dec[:,i]) è un vettore colonna D-dimensionale.
    # La norma ||W_dec_i||_2 è la norma L2 di questa colonna.
    # Lo sum sulle colonne ci dà un vettore (F,) con la norma L2 di ogni colonna di W_dec.
    W_dec_norms = torch.norm(W_dec, p=2, dim=0)
    
    # Calcola la penalizzazione: somma del prodotto delle attivazioni e delle norme
    l1_penalty = torch.mean(torch.matmul(feature_activations, W_dec_norms))

    # 5. Calcolo della loss totale
    total_loss = reconstruction_loss + lambd * l1_penalty

    return total_loss

def get_feature_directions(W_dec):
    """
    Calcola i vettori di direzione delle feature a partire dalla matrice dei pesi del decoder.

    Args:
        W_dec (torch.Tensor): Pesi del decoder di forma (D, F), dove
                              D è la dimensione residua e F la dimensione delle feature.

    Returns:
        torch.Tensor: I vettori delle feature (direzioni normalizzate), di forma (D, F).
    """
    # Calcola la norma L2 di ogni colonna (dim=0) della matrice W_dec.
    # Aggiunge 1e-8 per evitare divisioni per zero.
    norms = torch.norm(W_dec, p=2, dim=0, keepdim=True)
    
    # Normalizza ogni colonna (vettore di feature) dividendo per la sua norma.
    feature_directions = W_dec / (norms + 1e-8)
    
    return feature_directions

In [ ]:
from itertools import product
from tqdm import tqdm

os.makedirs("saved_models", exist_ok=True)

input_dim = 768  # Dimension of RoBERTa embeddings
hidden_dim = 768*8  # Dimension of the hidden layer in SAE

beta = 3.0      # Peso della penalizzazione
sae = SAE(input_dim, hidden_dim)

criterion = nn.MSELoss(reduction='sum')
optimizer = optim.Adam(sae.parameters(), lr=5e-5)

# Random data for demonstration
# Genera un dataset casuale di numeri
num_samples = 100  # Ridotto il numero di campioni per test più veloci
np.random.seed(42)


# Definisci gli iperparametri da testare
betas = np.arange(1.0, 5.5, 0.5)  # Da 1.0 a 5.0 con passo 0.5
lrs = np.arange(1e-5, 1.1e-4, 1e-5)  # Da 1e-5 a 1e-4 con passo 1e-5
best_loss = float('inf')
best_params = None

previous_results = []
with open("./saved_models/training_results.json", "r") as f:
    for l in f.readlines():
        previous_results.append(json.loads(l))
results = []
best_models = {}

for beta, lr in product(betas, lrs):
    for i, inputs in enumerate(activations):
        # Check if combination already performed
        found = False
        for trial in previous_results:
            if trial['layer'] == i and trial['hidden_dim'] == hidden_dim and trial["beta"] == beta and trial["lr"] == lr:
                found = True
                break
        # Skip training if already performed        
        if found:
            break
        X_train = inputs
        train_dataset = data.TensorDataset(torch.from_numpy(X_train))
        train_loader = data.DataLoader(train_dataset, batch_size=32, shuffle=True)  
        sae = SAE(input_dim, hidden_dim)
        optimizer = optim.Adam(sae.parameters(), lr=lr)
        epochs = 50
        for epoch in tqdm(range(epochs), desc=f"Layer {i} beta={beta} lr={lr}"):
            epoch_loss = 0
            for batch_idx, (data_batch,) in enumerate(train_loader):
                reconstructed_data, encoded_activations = sae(data_batch)
                encoder_weights = sae.encoder[0].weight
                decoder_weights = sae.decoder[0].weight
                loss = calculate_sae_loss(
                    data_batch, 
                    encoder_weights, 
                    sae.encoder[0].bias, 
                    decoder_weights, 
                    sae.decoder[0].bias, 
                    lambd=beta
                )
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            avg_loss = epoch_loss / len(X_train)
        print(f"Layer: {i}\t hidden_dim={hidden_dim}, beta={beta}, lr={lr}, avg_loss={avg_loss:.4f}")
        
        # Salva ogni risultato
        x = {"layer": i, "hidden_dim": hidden_dim, "beta": beta, "lr": lr, "avg_loss": avg_loss}
        with open("saved_models/training_results.json", "a") as f:
            json.dump(x, f)
        results.append(x)
        
        # Salva il modello migliore per ogni layer
        if i not in best_models or avg_loss < best_models[i]["loss"]:
            best_models[i] = {
                "model_state_dict": sae.state_dict(),
                "loss": avg_loss,
                "hidden_dim": hidden_dim,
                "beta": beta,
                "lr": lr
            }

# Salva i modelli migliori per ogni layer

for i, info in best_models.items():
    torch.save(info["model_state_dict"], f"saved_models/best_sae_layer_{i}.pt")

print("Best models saved in 'saved_models/' and training details in 'training_results.json'")

Layer 0 beta=1.0 lr=1e-05:   0%|          | 0/50 [00:32<?, ?it/s]


: 